# IT2011 — Artificial Intelligence & Machine Learning
## Progress Review I: Individual Preprocessing Notebook
**Student:** Athapaththu A. M. P. P.  
**IT Number:** `IT25102549`  
**Group ID:** `2026-Y2-S1-MTR-24`  
**Assigned Technique:** Text Cleaning, Noise Removal & Contraction Expansion  

---

### What This Notebook Covers
This notebook implements **Stage 1** of the group pipeline: raw text normalisation before any NLP model can consume the review text.

The pipeline removes structural noise (HTML, URLs), expands English contractions (`don't → do not`, `won't → will not`), strips punctuation, and collapses whitespace.  
The output column `cleaned_review` is handed to Member IT25102550 for tokenisation and lemmatisation.


## 0. Environment Setup

In [ ]:
import os, re, html, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

DATA_PATH = 'data/raw/Movies_Reviews_modified_version1.csv'

os.makedirs('results/eda_visualizations', exist_ok=True)
os.makedirs('results/outputs', exist_ok=True)

print("Libraries loaded. Output directories ready.")


## 1. Data Ingestion & Initial Inspection

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print(f"\nNull values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows          : {df.duplicated().sum()}")
print(f"Duplicate review texts  : {df['Reviews'].duplicated().sum()}")
df.head(3)


## 2. Assigned Technique — Text Cleaning, Noise Removal & Contraction Expansion

### Why Is This Needed?
Raw review text collected from the web contains several types of **noise** that are meaningless to a language model:

| Noise Type | Example | Why harmful |
|---|---|---|
| HTML entities / tags | `&amp;`, `<br>` | Adds non-linguistic tokens |
| URLs | `http://imdb.com/...` | No sentiment value |
| Contractions | `don't`, `won't` | Split into two tokens differently by different tokenisers; normalising to `do not` ensures consistency |
| Punctuation / digits | `!!!`, `2023` | Sparse, high-cardinality noise for bag-of-words |
| Mixed casing | `GREAT`, `great` | Same word treated as two vocabulary items |
| Extra whitespace | `"good  movie"` | Tokenisation artefacts |

Removing these ensures the vocabulary passed to the vectoriser consists **only** of meaningful English words.

### Contraction Map
A manually curated dictionary maps the most common English contractions to their expanded forms before any other transformation so that `n't`, `'re`, etc. are always expanded as complete words.


In [ ]:
# ── 2.1  Contraction dictionary ──────────────────────────────────────
CONTRACTIONS = {
    "won't":    "will not",
    "can't":    "cannot",
    "shan't":   "shall not",
    "n't":      " not",
    "'re":      " are",
    "'ve":      " have",
    "'ll":      " will",
    "'d":       " would",
    "'s":       " is",
    "it's":     "it is",
    "i'm":      "i am",
    "i've":     "i have",
    "i'll":     "i will",
    "i'd":      "i would",
    "you're":   "you are",
    "you've":   "you have",
    "you'll":   "you will",
    "he's":     "he is",
    "she's":    "she is",
    "we're":    "we are",
    "they're":  "they are",
    "that's":   "that is",
    "there's":  "there is",
    "what's":   "what is",
    "who's":    "who is",
    "how's":    "how is",
    "let's":    "let us",
    "could've": "could have",
    "would've": "would have",
    "should've":"should have",
    "might've": "might have",
}

print(f"Contraction dictionary contains {len(CONTRACTIONS)} entries.")
print("\nSample entries:")
for k, v in list(CONTRACTIONS.items())[:6]:
    print(f"  '{k}'  →  '{v}'")


In [ ]:
# ── 2.2  Full cleaning pipeline ──────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Pipeline:
      1. Guard against non-string input
      2. Decode HTML entities  (&amp; → &)
      3. Strip HTML / XML tags
      4. Remove URLs
      5. Lowercase
      6. Expand contractions
      7. Keep only [a-z] and whitespace
      8. Collapse multiple spaces
    """
    # 1. Guard
    if not isinstance(text, str):
        return ""

    # 2. HTML entity decode
    text = html.unescape(text)

    # 3. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 4. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 5. Lowercase
    text = text.lower()

    # 6. Expand contractions (longest-match first to avoid partial overlaps)
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)

    # 7. Strip non-alpha characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 8. Collapse whitespace
    return re.sub(r'\s+', ' ', text).strip()


# ── Apply ─────────────────────────────────────────────────────────────
df['cleaned_review'] = df['Reviews'].apply(clean_text)

print("Text cleaning complete.")
print(f"Rows processed: {len(df):,}")
print(f"\n{'─'*60}")
print("BEFORE:")
print(df['Reviews'].iloc[0][:300])
print(f"{'─'*60}")
print("AFTER:")
print(df['cleaned_review'].iloc[0][:300])


### 2.3 Contraction Expansion — Isolated Demo

In [ ]:
examples = [
    "I won't watch this movie again. It wasn't worth it.",
    "The director can't decide what kind of film he's making.",
    "I'd say it's the worst I've seen — don't waste your time.",
    "They're saying it could've been great, but it shouldn't have ended that way.",
]

print("=== Contraction Expansion Examples ===\n")
for raw in examples:
    cleaned = clean_text(raw)
    print(f"  BEFORE: {raw}")
    print(f"  AFTER : {cleaned}")
    print()


### 2.4 Derived Metrics — Word & Character Counts

In [ ]:
# Character counts
df['char_count_before'] = df['Reviews'].str.len()
df['char_count_after']  = df['cleaned_review'].str.len()

# Word counts
df['word_count_before'] = df['Reviews'].str.split().str.len()
df['word_count_after']  = df['cleaned_review'].str.split().str.len()

# Noise removed (%)
df['noise_pct'] = ((df['word_count_before'] - df['word_count_after'])
                   / df['word_count_before'] * 100).round(2)

print("Descriptive statistics — Word counts:")
print(df[['word_count_before','word_count_after','noise_pct']].describe().round(2))
print(f"\nAverage noise removed per review: {df['noise_pct'].mean():.1f}% of tokens")


## 3. EDA Visualisation — Word & Character Count Distributions: Before vs After Cleaning


In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

BEFORE_COL = '#4A90D9'
AFTER_COL  = '#2ECC71'

# ── Panel A: Word count histograms ────────────────────────────────────
ax_w_before = fig.add_subplot(gs[0, 0])
ax_w_after  = fig.add_subplot(gs[0, 1])

ax_w_before.hist(df['word_count_before'].clip(upper=600), bins=60,
                 color=BEFORE_COL, edgecolor='white', linewidth=0.4)
ax_w_before.set_title('Word Count BEFORE Cleaning', fontsize=13, fontweight='bold')
ax_w_before.set_xlabel('Words per review')
ax_w_before.set_ylabel('Number of reviews')
ax_w_before.axvline(df['word_count_before'].mean(), color='#E74C3C',
                    linestyle='--', linewidth=1.8, label=f"Mean = {df['word_count_before'].mean():.0f}")
ax_w_before.legend(fontsize=10)

ax_w_after.hist(df['word_count_after'].clip(upper=600), bins=60,
                color=AFTER_COL, edgecolor='white', linewidth=0.4)
ax_w_after.set_title('Word Count AFTER Cleaning', fontsize=13, fontweight='bold')
ax_w_after.set_xlabel('Words per review')
ax_w_after.set_ylabel('Number of reviews')
ax_w_after.axvline(df['word_count_after'].mean(), color='#E74C3C',
                   linestyle='--', linewidth=1.8, label=f"Mean = {df['word_count_after'].mean():.0f}")
ax_w_after.legend(fontsize=10)

# ── Panel B: Character count histograms ───────────────────────────────
ax_c_before = fig.add_subplot(gs[1, 0])
ax_c_after  = fig.add_subplot(gs[1, 1])

ax_c_before.hist(df['char_count_before'].clip(upper=4000), bins=60,
                 color=BEFORE_COL, edgecolor='white', linewidth=0.4, alpha=0.85)
ax_c_before.set_title('Character Count BEFORE Cleaning', fontsize=13, fontweight='bold')
ax_c_before.set_xlabel('Characters per review')
ax_c_before.set_ylabel('Number of reviews')
ax_c_before.axvline(df['char_count_before'].mean(), color='#E74C3C',
                    linestyle='--', linewidth=1.8, label=f"Mean = {df['char_count_before'].mean():.0f}")
ax_c_before.legend(fontsize=10)

ax_c_after.hist(df['char_count_after'].clip(upper=4000), bins=60,
                color=AFTER_COL, edgecolor='white', linewidth=0.4, alpha=0.85)
ax_c_after.set_title('Character Count AFTER Cleaning', fontsize=13, fontweight='bold')
ax_c_after.set_xlabel('Characters per review')
ax_c_after.set_ylabel('Number of reviews')
ax_c_after.axvline(df['char_count_after'].mean(), color='#E74C3C',
                   linestyle='--', linewidth=1.8, label=f"Mean = {df['char_count_after'].mean():.0f}")
ax_c_after.legend(fontsize=10)

fig.suptitle(
    'IT25102549 — Text Cleaning EDA\n'
    'Word & Character Count Distributions: Before vs After Cleaning',
    fontsize=15, fontweight='bold', y=1.01
)

OUT_PATH = 'results/eda_visualizations/member1_text_cleaning_distributions.png'
plt.savefig(OUT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"EDA plot saved → {OUT_PATH}")


## 4. Save Cleaned Dataset

In [ ]:
cols_to_keep = ['Ratings', 'cleaned_review', 'movie_name', 'Resenhas',
                'genres', 'Description', 'emotion',
                'char_count_before', 'char_count_after',
                'word_count_before', 'word_count_after', 'noise_pct']
df_out = df[[c for c in cols_to_keep if c in df.columns]]

OUT_CSV = 'results/outputs/stage1_IT25102549_cleaned.csv'
df_out.to_csv(OUT_CSV, index=False)

print(f"Stage 1 output saved → {OUT_CSV}")
print(f"Shape: {df_out.shape}")
print(f"\nColumn list: {list(df_out.columns)}")
print(f"\nSample cleaned_review:")
print(df_out['cleaned_review'].iloc[2][:250])
